# Burp Suite: A Practical Learning Guide

> **Level:** Beginner to intermediate  
> **Objective:** Understand Burp Suite and use it to investigate web application and API security.  
> **Practice environment:** Your own application or an authorized training lab.  
> **Tools:** Burp Suite Community Edition and PortSwigger Web Security Academy.

---

##  Table of Contents

1. Introduction
2. HTTP Fundamentals
3. Authentication and Authorization
4. Burp Setup
5. Burp Tools
6. Proxy
7. Target and Scope
8. Repeater
9. Intruder
10. Decoder
11. Comparer
12. Sequencer
13. Scanner
14. Collaborator
15. Web Vulnerabilities
16. API Security
17. Sessions and Request Sequences
18. WebSockets
19. AI Application Testing
20. Testing Methodology
21. Reporting and Retesting
22. Troubleshooting
23. Practical Exercises
24. Learning Resources

---

## 1. Introduction

### What Is Burp Suite?

Burp Suite is a toolkit for testing websites, web applications, and APIs.

It helps you:

- Capture browser requests.
- Inspect server responses.
- Modify application inputs.
- Resend requests.
- Compare application behavior.
- Investigate whether security rules are enforced.

### How It Works

A browser sends a request to an application. The application processes the request and returns a response.

Burp Proxy sits between them and inspects traffic routed through it.

### Example

```http
POST /api/profile HTTP/1.1
Host: training.example
Content-Type: application/json
Cookie: session=sample-session

{
  "display_name": "Sharath"
}
```

A tester asks:

> Does the server allow this user to change only permitted profile fields?

Changing a request is an experiment. A vulnerability requires evidence that a security rule was violated.

---

## 2. HTTP Fundamentals

### 2.1 Request Structure

An HTTP/1.1 request contains:

1. Request line.
2. Headers.
3. Blank line.
4. Optional body.

```http
POST /api/orders HTTP/1.1
Host: training.example
Content-Type: application/json
Cookie: session=sample-session

{
  "product_id": 15,
  "quantity": 2
}
```

| Component | Meaning |
|---|---|
| POST | HTTP method |
| /api/orders | Endpoint |
| Host | Destination application |
| Content-Type | Request body format |
| Cookie | Cookie supplied by the client |
| JSON body | Submitted application data |

> These examples use HTTP/1.1 notation. Modern protocols can have different on-wire representations.

### 2.2 HTTP Methods

| Method | Typical Purpose |
|---|---|
| GET | Retrieve information |
| POST | Submit data or perform an operation |
| PUT | Replace a resource |
| PATCH | Update selected fields |
| DELETE | Remove a resource |
| HEAD | Retrieve headers without response content |
| OPTIONS | Describe communication options |

The method does not guarantee what the application actually does.

### 2.3 Response Structure

```http
HTTP/1.1 201 Created
Content-Type: application/json

{
  "order_id": 501,
  "status": "created"
}
```

### 2.4 Status Codes

| Code | General Meaning |
|---|---|
| 200 | Request succeeded |
| 201 | Resource created |
| 204 | Success without response content |
| 301 / 302 | Redirect |
| 400 | Invalid request |
| 401 | Missing or invalid authentication |
| 403 | Access forbidden |
| 404 | Resource not found |
| 409 | Conflict with current state |
| 429 | Too many requests |
| 500 | Server error |

**Important:** Read the response body and verify application state. A `200` response does not automatically mean an operation succeeded.

### 2.5 Input Locations

| Location | Example |
|---|---|
| URL path | /api/invoices/101 |
| Query string | /search?q=tax |
| Form body | username=alice |
| JSON body | {"document_id":"doc-101"} |
| Cookie | session=sample-session |
| Header | X-Workspace-ID: workspace-1 |
| File upload | Filename and file content |
| WebSocket message | {"action":"send","text":"hello"} |

For every input, ask what it controls and who should be allowed to choose its value.

---

## 3. Authentication and Authorization

### Authentication

Authentication establishes identity.

**Question:** Who is making the request?

Examples:

- Password.
- Passkey.
- One-time code.
- Identity provider.

### Authorization

Authorization determines permitted actions.

**Question:** May this user perform this operation on this resource?

Examples:

- A user reads their own invoice.
- A viewer reads a document.
- An administrator manages users.

### Sessions

A session connects subsequent requests to an authenticated identity.

```http
Set-Cookie: session=sample-session; Secure; HttpOnly; SameSite=Lax
```

Later:

```http
Cookie: session=sample-session
```

| Attribute | Purpose |
|---|---|
| Secure | Restricts transmission to secure connections |
| HttpOnly | Prevents normal JavaScript access |
| SameSite | Controls certain cross-site cookie behavior |
| Domain | Defines eligible hosts |
| Path | Defines eligible paths |
| Expires / Max-Age | Controls persistence |

### JWT Basics

A JWT commonly contains:

```text
header.payload.signature
```

The header and payload are generally encoded rather than encrypted.

Decoding a JWT does not prove that you can forge a token accepted by the server.

---

## 4. Burp Setup

### Local Installation

1. Download Burp from the official PortSwigger website.
2. Select the installer for your operating system and processor.
3. Install and launch Burp.
4. Start Community Edition.
5. Use the default starting configuration.
6. Open the built-in browser.

```text
Proxy → Intercept → Open browser
```

### Why Use the Built-In Browser?

It is configured for Burp’s proxy and HTTPS testing.

### HTTPS Interception

For HTTPS inspection:

- The browser establishes a secure connection with Burp.
- Burp establishes another secure connection with the target.
- The browser trusts Burp’s testing certificate authority.

An external browser requires appropriate proxy and certificate configuration.

> Use Kaggle for learning notes and analysis. Run the Burp desktop application locally.

---

## 5. Burp Tools

| Tool | Purpose |
|---|---|
| Proxy | Capture and intercept traffic |
| Target | Organize resources and scope |
| Repeater | Edit and resend individual requests |
| Intruder | Repeat requests with varied inputs |
| Decoder | Transform encoded data |
| Comparer | Compare messages |
| Sequencer | Analyze token randomness |
| Scanner | Automate discovery and vulnerability testing |
| Collaborator | Observe external callbacks |
| Extensions | Add functionality |

### Editions

Community Edition provides essential manual tools and limited Intruder functionality.

Professional adds features including:

- Automated scanning.
- Full Intruder.
- Collaborator.
- Automatic crawling.
- Persistent project files.

---

## 6. Proxy

### HTTP History

HTTP history records traffic passing through Burp.

Start by browsing with interception off.

Inspect:

- Method.
- Endpoint.
- Parameters.
- Cookies.
- Response code.
- Response body.
- Redirects.

One browser action may generate many requests. Identify the request responsible for the operation.

### Interception

Interception pauses matching traffic.

| Action | Effect |
|---|---|
| Forward | Sends the request onward |
| Drop | Prevents delivery |
| Edit | Changes the request before forwarding |

### Example

```http
GET /products?category=Books HTTP/1.1
Host: training.example
```

Change:

```text
category=Books
```

To:

```text
category=Electronics
```

This demonstrates input modification, not necessarily a vulnerability.

**Troubleshooting:** A hanging browser may have a request waiting in Intercept.

---

## 7. Target and Scope

### Site Map

The site map organizes observed resources.

```text
/
/login
/account
/api/profile
/api/invoices
```

### Scope

Scope identifies intended testing hosts and paths.

1. Add the lab host.
2. Filter history to relevant traffic.
3. Review each tool’s scope behavior.

Scope settings are organizational controls, not a universal network barrier.

---

## 8. Repeater

Repeater is used for controlled manual experiments.

### Workflow

1. Select a request in HTTP history.
2. Send it to Repeater.
3. Send it unchanged.
4. Confirm the baseline.
5. Change one input.
6. Resend.
7. Compare the result.

### Example: Object Authorization

Lab accounts:

| User | Private Invoice |
|---|---|
| Alice | 101 |
| Bob | 102 |

Original:

```http
GET /api/invoices/101 HTTP/1.1
Host: training.example
Cookie: session=alice-test-session
```

Modified:

```http
GET /api/invoices/102 HTTP/1.1
Host: training.example
Cookie: session=alice-test-session
```

**Security rule:** Alice must not receive Bob’s private invoice.

| Result | Interpretation |
|---|---|
| Access denied | Consistent with authorization enforcement |
| Bob’s private invoice returned | Evidence of broken authorization |
| Login page | Session may be invalid |
| Error | Requires investigation |

### Browser Limitations

Repeater does not enforce browser controls in the same way as a browser.

Browser-dependent findings involving CORS, SameSite, or JavaScript require browser validation.

---

## 9. Intruder

Intruder replaces selected positions with payload values.

```http
GET /api/products/§1§ HTTP/1.1
Host: training.example
```

Payloads:

```text
1
2
3
4
5
```

The markers indicate insertion positions in Burp’s interface.

### Attack Types

| Type | Behavior |
|---|---|
| Sniper | Changes positions separately |
| Battering ram | Uses one value across all positions |
| Pitchfork | Pairs corresponding list entries |
| Cluster bomb | Tests combinations across lists |

### Analyze Results

Review:

- Status.
- Length.
- Messages.
- Redirects.
- Timing.
- Actual state.

Repeat unusual results manually in Repeater.

---

## 10. Decoder

### Examples

```text
Hello%20Sharath → Hello Sharath
```

```text
c2hhcmF0aA== → sharath
```

### Important Distinctions

| Concept | Purpose |
|---|---|
| Encoding | Changes representation |
| Encryption | Protects confidentiality using a key |
| Hashing | Produces a digest |

Base64 is not encryption.

A decoded value may still be protected by a signature or other integrity check.

---

## 11. Comparer

Comparer identifies differences between messages.

### Workflow

1. Send two messages to Comparer.
2. Select both.
3. Compare words or bytes.
4. Inspect meaningful differences.

Ignore unrelated changes such as:

- Timestamps.
- Request IDs.
- Rotating tokens.
- Dynamic content.

---

## 12. Sequencer

Sequencer statistically examines token samples.

Its purpose is to investigate predictability.

A strong statistical result does not establish complete session security.

Also evaluate:

- Expiration.
- Rotation.
- Logout invalidation.
- Secure transmission.
- Token exposure.
- Server-side handling.

---

## 13. Scanner

### Passive Analysis

Examines observed traffic.

### Active Testing

Sends additional requests to investigate weaknesses.

Active testing can change application state.

### Limitations

A scanner may:

- Lose authentication.
- Miss workflows.
- Misinterpret responses.
- Miss business logic.
- Report false positives.

Manually verify findings.

An empty scan does not prove security.

---

## 14. Collaborator

Collaborator records interactions with unique callback addresses.

### Workflow

1. Generate an address.
2. Insert it into an authorized test.
3. Submit the request.
4. Check interactions.
5. Interpret the evidence.

A callback establishes that an interaction occurred.

It does not automatically establish that a security boundary was violated.

---

## 15. Web Vulnerabilities

| Vulnerability | Core Problem |
|---|---|
| Broken access control | Unauthorized access or action |
| Authentication weakness | Incorrect identity or session enforcement |
| SQL injection | Input changes database query structure |
| XSS | Untrusted content executes script |
| CSRF | Another site causes an unwanted authenticated action |
| Path traversal | File access escapes its permitted location |
| SSRF | Server-side requests cross intended boundaries |
| File upload weakness | Uploaded content is handled unsafely |
| Business logic flaw | Workflow violates a business rule |
| Race condition | Concurrent operations violate a state rule |

### Testing Principles

- Identify expected behavior.
- Use controlled lab inputs.
- Validate observations.
- Distinguish clues from proof.
- Explain demonstrated impact.

---

## 16. API Security

Example:

```http
PATCH /api/users/me HTTP/1.1
Host: training.example
Content-Type: application/json
Cookie: session=sample-session

{
  "display_name": "Sharath"
}
```

### Checklist

| Area | Question |
|---|---|
| Authentication | Is identity required? |
| Object authorization | May the user access this object? |
| Function authorization | May the user perform this action? |
| Field authorization | May the user modify this field? |
| Validation | Are types and values checked? |
| State | Is this operation valid now? |
| Exposure | Is unnecessary private data returned? |
| Resources | Are expensive operations constrained? |

### Mass Assignment

A backend may improperly accept fields that the user should not control.

The backend should explicitly permit editable fields and independently enforce authorization.

---

## 17. Sessions and Request Sequences

A request may depend on:

- A valid session.
- A fresh CSRF token.
- A preceding request.
- A particular state.
- A single-use value.

Before interpreting a failed request, verify these conditions.

Learn the manual workflow before automating token refresh or session handling.

---

## 18. WebSockets

Example message:

```json
{
  "action": "send_message",
  "room_id": "room-1",
  "text": "Hello"
}
```

Investigate:

- Room authorization.
- Per-action permissions.
- Safe content handling.
- Logout behavior.
- Permission changes.
- Client-supplied identity fields.

An authenticated handshake does not prove authorization for every later operation.

---

## 19. AI Application Testing

Example:

```http
POST /api/chat HTTP/1.1
Host: training.example
Content-Type: application/json
Cookie: session=alice-test-session

{
  "conversation_id": "conversation-101",
  "document_id": "document-101",
  "message": "Summarize this document."
}
```

### Boundaries to Test

| Boundary | Evidence |
|---|---|
| User to conversation | Unauthorized conversation returned |
| User to document | Unauthorized document accessed |
| User to privileges | Unauthorized capability granted |
| Document to instructions | Untrusted instructions followed |
| Model to tools | Prohibited tool operation executed |

Use synthetic canaries and explicit expected behavior.

Burp captures HTTP evidence. Backend logs may be needed to confirm retrieval or tool execution.

Model text claiming an action occurred is not proof that it occurred.

---

## 20. Testing Methodology

1. Define the security rule.
2. Observe normal behavior.
3. Capture a baseline.
4. Form a hypothesis.
5. Change one variable.
6. Verify the result.
7. Record evidence.
8. Retest after remediation.

### Example Rule

> A viewer may read a document but may not delete it.

The experiment should specifically test that rule.

---

## 21. Reporting and Retesting

### Report Fields

| Field | Content |
|---|---|
| Title | Concrete problem |
| Feature | Endpoint and operation |
| Preconditions | Account role and state |
| Expected behavior | Security rule |
| Steps | Reproducible sequence |
| Actual behavior | Observed violation |
| Evidence | Requests, responses, and state |
| Impact | Demonstrated consequence |
| Remediation | Required enforcement |
| Retest | Result after the fix |

Redact credentials and session secrets.

### Retest Procedure

- Repeat legitimate behavior.
- Repeat the original failing test.
- Confirm unauthorized behavior is prevented.
- Confirm legitimate behavior still works.
- Record version and test conditions.

---

## 22. Troubleshooting

| Problem | Check |
|---|---|
| Browser hangs | Intercepted request waiting |
| No traffic | Browser configuration and history filters |
| HTTPS warning | Certificate trust |
| Login page in Repeater | Session validity |
| Request works once | Token reuse or state change |
| Identical Intruder results | Payload positions and baseline |
| Random response differences | Dynamic content |
| Scanner misses a feature | Authentication and workflow coverage |

---

## 23. Practical Exercises

### Exercise 1: Read a Request

Identify:

- Method.
- Endpoint.
- Headers.
- Authentication.
- Inputs.
- Response status.
- Response content.

### Exercise 2: Repeater Experiment

1. Capture a request.
2. Send it unchanged.
3. Modify one harmless input.
4. Explain the result.

### Exercise 3: Authorization Lab

Use two test accounts with known private resources.

Verify whether each account can access only permitted resources.

### Observation Template

```text
Feature:

Security rule:

Baseline request:

Baseline result:

Input changed:

Observed result:

Verified state:

Conclusion:

Remaining uncertainty:
```

### Completion Criteria

You can explain:

- What was tested.
- Why the input mattered.
- What changed.
- Whether a security rule was violated.
- What the evidence does not establish.



